# CLIP — zero-shot classification vs. a trained linear probe

**Foundation model** · supports lessons C4 (recognition) and D4 (open-vocabulary semantics).

CLIP is a pretrained vision-language model. We'll (1) use it **zero-shot** — classify images by comparing them to text prompts, no training — then (2) **train a linear probe** on its frozen image features and compare. This is the canonical way to *apply* a foundation model: freeze the backbone, train a small head.

> **Runtime → T4 GPU** recommended. This notebook is meant to run on Colab (it downloads CLIP weights + a small dataset).

In [ ]:
!pip -q install open_clip_torch
import torch, open_clip, random, numpy as np, torch.nn as nn
from torchvision import datasets
device = "cuda" if torch.cuda.is_available() else "cpu"
model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
model = model.to(device).eval(); tokenizer = open_clip.get_tokenizer("ViT-B-32")
print("CLIP ready on", device)

## 1 · A small labeled image set (CIFAR-10 test split)

In [ ]:
classes = ["airplane","automobile","bird","cat","deer","dog","frog","horse","ship","truck"]
test = datasets.CIFAR10(".", train=False, download=True)
random.seed(0); idxs = random.sample(range(len(test)), 800)

## 2 · Zero-shot — no training
Encode each class as "a photo of a {class}", encode each image, pick the nearest text.

In [ ]:
text = tokenizer([f"a photo of a {c}" for c in classes]).to(device)
with torch.no_grad():
    tf = model.encode_text(text); tf = tf / tf.norm(dim=-1, keepdim=True)
feats, labels, zs_correct = [], [], 0
with torch.no_grad():
    for i in idxs:
        img, lab = test[i]
        f = model.encode_image(preprocess(img).unsqueeze(0).to(device))
        f = f / f.norm(dim=-1, keepdim=True)
        feats.append(f.cpu()); labels.append(lab)
        zs_correct += int((f @ tf.T).argmax().item() == lab)
zs_acc = zs_correct / len(idxs)
print(f"zero-shot accuracy: {zs_acc:.3f}")

## 3 · Linear probe — train a head on frozen features
Features are already extracted (the backbone never updates). We only train a 10-way linear layer.

In [ ]:
X = torch.cat(feats); y = torch.tensor(labels)
ntr = 600
Xtr, ytr, Xte, yte = X[:ntr], y[:ntr], X[ntr:], y[ntr:]
clf = nn.Linear(X.shape[1], 10)
opt = torch.optim.Adam(clf.parameters(), 1e-3)
for epoch in range(400):
    opt.zero_grad(); loss = nn.functional.cross_entropy(clf(Xtr), ytr); loss.backward(); opt.step()
probe_acc = (clf(Xte).argmax(-1) == yte).float().mean().item()
print(f"linear-probe accuracy: {probe_acc:.3f}")

## 4 · Compare

In [ ]:
import matplotlib.pyplot as plt
plt.bar(["zero-shot", "linear probe"], [zs_acc, probe_acc], color=["C7", "C0"])
plt.ylabel("accuracy"); plt.ylim(0, 1); plt.title("CLIP: applying a foundation model")
for i, v in enumerate([zs_acc, probe_acc]): plt.text(i, v + .02, f"{v:.2f}", ha="center")
plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/CD_clip_zeroshot_probe/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/CD_clip_zeroshot_probe"; os.makedirs(run, exist_ok=True)
torch.save(clf.state_dict(), f"{run}/probe.pt")
json.dump({"zero_shot": zs_acc, "linear_probe": probe_acc}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

### Where to go next
- Swap prompts ("a blurry photo of a {c}", ensembles) to boost zero-shot.
- Use CLIP features for **open-vocabulary** segmentation/detection → **OpenScene / LERF** (lesson D4).
- For frozen-feature *video* recognition, the next lab fine-tunes **VideoMAE**.